In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS churn_lakehouse.semantic;

In [0]:
%sql
CREATE OR REPLACE VIEW churn_lakehouse.semantic.sem_churn_overview AS
SELECT
    -- Volume
    COUNT(*)                                                AS total_customers,
    SUM(CASE WHEN c.churned THEN 1 ELSE 0 END)             AS total_churned,
    SUM(CASE WHEN NOT c.churned THEN 1 ELSE 0 END)         AS total_active,

    -- Churn rate real (histórico)
    ROUND(AVG(CASE WHEN c.churned THEN 1.0 ELSE 0.0 END) * 100, 2)
                                                           AS churn_rate_pct,

    -- Churn rate predito pelo modelo
    ROUND(AVG(p.churn_probability) * 100, 2)               AS predicted_churn_rate_pct,

    -- MRR
    ROUND(SUM(CASE WHEN NOT c.churned THEN c.current_mrr_brl ELSE 0 END), 2)
                                                           AS total_active_mrr,
    ROUND(SUM(p.churn_probability * c.current_mrr_brl), 2) AS expected_mrr_loss,

    -- Segmentos de risco
    SUM(CASE WHEN p.ml_risk_segment = 'high'   THEN 1 ELSE 0 END) AS high_risk_customers,
    SUM(CASE WHEN p.ml_risk_segment = 'medium' THEN 1 ELSE 0 END) AS medium_risk_customers,
    SUM(CASE WHEN p.ml_risk_segment = 'low'    THEN 1 ELSE 0 END) AS low_risk_customers,

    -- MRR em risco por segmento
    ROUND(SUM(CASE WHEN p.ml_risk_segment = 'high'
              THEN c.current_mrr_brl ELSE 0 END), 2)       AS high_risk_mrr,
    ROUND(SUM(CASE WHEN p.ml_risk_segment = 'medium'
              THEN c.current_mrr_brl * 0.4 ELSE 0 END), 2) AS medium_risk_mrr

FROM churn_lakehouse.gold.gold_customer_360 c
JOIN churn_lakehouse.gold.gold_churn_predictions p
  ON c.customer_id = p.customer_id;

In [0]:
%sql
CREATE OR REPLACE VIEW churn_lakehouse.semantic.sem_revenue_at_risk AS
SELECT
    c.plan,
    p.ml_risk_segment                                       AS risk_segment,
    COUNT(*)                                                AS customers,
    ROUND(SUM(c.current_mrr_brl), 2)                       AS total_mrr,
    ROUND(SUM(p.churn_probability * c.current_mrr_brl), 2) AS expected_mrr_loss,
    ROUND(AVG(p.churn_probability) * 100, 2)               AS avg_churn_prob_pct,
    ROUND(
        SUM(p.churn_probability * c.current_mrr_brl) /
        NULLIF(SUM(c.current_mrr_brl), 0) * 100
    , 2)                                                    AS mrr_at_risk_pct

FROM churn_lakehouse.gold.gold_customer_360 c
JOIN churn_lakehouse.gold.gold_churn_predictions p
  ON c.customer_id = p.customer_id
WHERE NOT c.churned
GROUP BY c.plan, p.ml_risk_segment
ORDER BY expected_mrr_loss DESC;

In [0]:
%sql
CREATE OR REPLACE VIEW churn_lakehouse.semantic.sem_cohort_analysis AS
SELECT
    c.acquisition_cohort,
    COUNT(*)                                                AS total_customers,
    SUM(CASE WHEN c.churned THEN 1 ELSE 0 END)             AS churned_customers,
    ROUND(AVG(CASE WHEN c.churned THEN 1.0 ELSE 0.0 END) * 100, 2)
                                                           AS churn_rate_pct,
    ROUND(AVG(c.tenure_days), 0)                           AS avg_tenure_days,
    ROUND(AVG(p.churn_probability) * 100, 2)               AS avg_predicted_churn_pct,
    ROUND(SUM(c.current_mrr_brl), 2)                       AS total_mrr,
    ROUND(SUM(p.churn_probability * c.current_mrr_brl), 2) AS expected_mrr_loss

FROM churn_lakehouse.gold.gold_customer_360 c
JOIN churn_lakehouse.gold.gold_churn_predictions p
  ON c.customer_id = p.customer_id
GROUP BY c.acquisition_cohort
ORDER BY c.acquisition_cohort;

In [0]:
%sql
CREATE OR REPLACE VIEW churn_lakehouse.semantic.sem_customer_health AS
SELECT
    c.customer_id,
    c.company_name,
    c.plan,
    c.industry,
    c.acquisition_cohort,
    c.tenure_months,
    c.current_mrr_brl,
    c.nps_score,
    c.nps_category,
    c.engagement_level,
    c.recommended_action,
    p.churn_probability,
    p.ml_risk_segment                                       AS risk_segment,
    p.shap_reason_1                                         AS top_churn_reason,
    p.shap_reason_2                                         AS second_churn_reason,

    -- Health score composto (inverso da probabilidade de churn)
    ROUND((1 - p.churn_probability) * 100, 1)              AS health_score,

    -- Classificação de health
    CASE
        WHEN p.churn_probability < 0.30 THEN 'Saudável'
        WHEN p.churn_probability < 0.60 THEN 'Atenção'
        ELSE 'Crítico'
    END                                                     AS health_status,

    -- Revenue at risk individual
    ROUND(p.churn_probability * c.current_mrr_brl, 2)      AS individual_mrr_at_risk

FROM churn_lakehouse.gold.gold_customer_360 c
JOIN churn_lakehouse.gold.gold_churn_predictions p
  ON c.customer_id = p.customer_id
WHERE NOT c.churned
ORDER BY p.churn_probability DESC;

In [0]:
%sql
SELECT 'churn_overview' AS view_name, COUNT(*) AS rows
FROM churn_lakehouse.semantic.sem_churn_overview
UNION ALL
SELECT 'revenue_at_risk', COUNT(*)
FROM churn_lakehouse.semantic.sem_revenue_at_risk
UNION ALL
SELECT 'cohort_analysis', COUNT(*)
FROM churn_lakehouse.semantic.sem_cohort_analysis
UNION ALL
SELECT 'customer_health', COUNT(*)
FROM churn_lakehouse.semantic.sem_customer_health;

view_name,rows
churn_overview,1
revenue_at_risk,12
cohort_analysis,18
customer_health,1455


In [0]:
%sql
-- Overview executivo
SELECT * FROM churn_lakehouse.semantic.sem_churn_overview;

total_customers,total_churned,total_active,churn_rate_pct,predicted_churn_rate_pct,total_active_mrr,expected_mrr_loss,high_risk_customers,medium_risk_customers,low_risk_customers,high_risk_mrr,medium_risk_mrr
2000,545,1455,27.25,27.5,1583145.0,580534.54,478,211,1311,505122.0,90595.6
